# Opening Line — Whisper demo (Google Colab)

This notebook runs entirely in Google Colab. It lets you upload an audio file and transcribe it using the `tiny` Whisper model.

Steps for nontechnical users: run the cells in order, upload an audio file, and download the generated `.txt` transcription.

In [1]:
# Install dependencies (Colab). This may take a minute or two.
!pip install -q openai-whisper ffmpeg-python
print('Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.8 MB/s eta 0:00:00
Dependencies installed


Record from your browser and download transcribed text. Click the button and allow microphone access.

In [7]:
import whisper
from IPython.display import display, HTML, Javascript
from google.colab import output
import base64
import re


model = whisper.load_model("tiny")

def receive_audio(data):
    audio = re.sub(r'^data:audio/\w+;base64,', '', data)

    with open("recording.webm", "wb") as f:
        f.write(base64.b64decode(audio))

    print("Saved recording.webm")

    result = model.transcribe("recording.webm")
    print(result["text"])

output.register_callback("notebook.receive_audio", receive_audio)


display(HTML("""
<button id="recordBtn" onclick="toggleRecording()"
        style="font-size:18px;padding:12px 24px;">
🎙️ Start Recording
</button>
"""))

display(Javascript("""
let mediaRecorder = null;
let audioChunks = [];
let stream = null;
let isRecording = false;

async function toggleRecording() {

  const btn = document.getElementById("recordBtn");

  if (!isRecording) {

    try {
      stream = await navigator.mediaDevices.getUserMedia({ audio: true });

      mediaRecorder = new MediaRecorder(stream);
      audioChunks = [];

      mediaRecorder.ondataavailable = event => {
        if (event.data.size > 0) {
          audioChunks.push(event.data);
        }
      };

      mediaRecorder.start();

      isRecording = true;

      btn.innerHTML = "⏹️ Stop Recording";

      console.log("Recording started");

    } catch (err) {
      alert("Microphone access failed: " + err.message);
      console.error(err);
    }

  } else {

    mediaRecorder.stop();

    await new Promise(resolve => {
      mediaRecorder.onstop = resolve;
    });

    stream.getTracks().forEach(track => track.stop());

    const blob = new Blob(audioChunks, {
      type: 'audio/webm'
    });

    const reader = new FileReader();

    reader.onloadend = () => {
      google.colab.kernel.invokeFunction(
        'notebook.receive_audio',
        [reader.result],
        {}
      );
    };

    reader.readAsDataURL(blob);

    isRecording = false;

    btn.innerHTML = "🎙️ Start Recording";

    console.log("Recording stopped");
  }
}
"""))

100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 79.3MiB/s]


<IPython.core.display.Javascript object>

Saved recording.webm


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 A 37-year-old female presents to the office accompanied by her partner for fallowendometriosis. She has had chronic pelvic pain for several years. She takes oral contraceptives continuously to suppress her menstrual cycle and non-stereotrial medications. And hydrocordin, acetaminophen, has needed for pain and relief. At her last visit, the physician and patient discussed possible treatments. Included surgical ablation of the patient. At that time, the patient felt that the pain was not severe enough to warrant a surgical procedure. Today, after the physician enters the room and introduces herself to the patient's partner, says she is having more pain and sometimes cannot get out of bed. Something else has to be done. The most appropriate action by the physician is to...
